# CS348K — V-Scale Full Pipeline Notebook
**Run this top to bottom after a fresh clone.**

```
Section 1  — Setup
Section 2  — Check configs
Section 2b — Fix experimental design (one variable at a time)
Section 3  — Run LTX sweep (unoptimized)
Section 4  — Run CogVideoX sweep (unoptimized)
Section 5  — Run V-Scale eval (Pareto + budget selections)
Section 6  — Identify bottlenecks from scaling data
             → observation leads to hypothesis leads to optimization choice
Section 7  — Add optimization configs (justified by Section 6 findings)
Section 8  — Re-run LTX sweep (optimized configs only)
Section 9  — Re-run CogVideoX sweep (optimized configs only)
Section 10 — Re-run V-Scale eval (new Pareto frontier)
Section 11 — Compare: did optimization change the frontier?
Section 12 — All charts (bottleneck + optimization + Pareto)
Section 13 — Update presentation slides
Section 14 — Print final transcript numbers
```

---
## Section 1 — Setup

In [ ]:
import os, subprocess, sys

REPO = "visual_computing_systems"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone",
        "https://github.com/jennyyjin/visual_computing_systems.git"], check=True)
    print("Cloned.")
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
    print("Already cloned — pulled latest.")

os.chdir(REPO)
print("Working directory:", os.getcwd())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install",
    "modal", "pandas", "matplotlib", "python-pptx",
    "seaborn", "torchao", "--quiet"], check=True)
print("Dependencies installed.")

In [ ]:
# Run `python -m modal setup` in the VS Code terminal first (one-time).
# This cell just verifies authentication worked.
result = subprocess.run(
    [sys.executable, "-m", "modal", "profile", "current"],
    capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Modal authenticated:", result.stdout.strip())
else:
    print("✗ Not authenticated — run `python -m modal setup` in the terminal first")

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "modal", "volume", "create", "vscale-hf-cache"],
    capture_output=True, text=True)
print(result.stdout or result.stderr)

---
## Section 2 — Check Existing Configs

In [ ]:
import json

with open("configs/modal_sweep.json") as f:
    configs = json.load(f)

print(f"Total configs: {len(configs)}\n")
print(f"{'config_id':<48} {'backend':<12} {'steps':>5}  {'WxH':>10}  {'frames':>6}")
print("-" * 90)
for c in sorted(configs, key=lambda x: (x['backend'], x['config_id'])):
    print(f"  {c['config_id']:<46} {c['backend']:<12} {c['steps']:>5}  "
          f"{c['width']}x{c['height']:>3}  {c['frames']:>6}")

---
## Section 2b — Fix Experimental Design

**Rule: change ONE variable at a time.**

- **Step sweep**: fix resolution + frames, vary steps
- **Frame sweep**: fix resolution + steps, vary frames
  - 17f = ~2s, 33f = ~4s, 49f = ~6s (baseline), 81f = ~10s
  - All pass validation: LTX needs 8k+1, CogX needs 4k+1
- **Resolution sweep**: fix steps + frames, vary resolution

This gives us clean data to identify bottlenecks in Section 6.

In [ ]:
with open("configs/modal_sweep.json") as f:
    configs = json.load(f)

# Remove any old 25f configs added previously
configs = [c for c in configs if "25f" not in c["config_id"]]
existing = {c["config_id"] for c in configs}

ltx_base = next(c for c in configs if c["config_id"] == "ltx_fast_384x640_49f_20s")
cog_base = next(c for c in configs if c["config_id"] == "cogvideox_fast_480x768_49f_15s")

new = []

# ── LTX step sweep (fix 384x640, 49 frames) ──────────────────────────────────
# already have: 4s, 8s, 15s, 20s — add 30s to complete the range
new.append({**ltx_base, "config_id": "ltx_fast_384x640_49f_30s", "steps": 30})

# ── LTX frame sweep (fix 384x640, 20 steps) ──────────────────────────────────
# 17=8*2+1, 33=8*4+1, 49=baseline (exists), 81=8*10+1
new.append({**ltx_base, "config_id": "ltx_fast_384x640_17f_20s", "frames": 17})
new.append({**ltx_base, "config_id": "ltx_fast_384x640_33f_20s", "frames": 33})
new.append({**ltx_base, "config_id": "ltx_fast_384x640_81f_20s", "frames": 81})

# ── LTX resolution sweep (fix 20 steps, 49 frames) ───────────────────────────
new.append({**ltx_base, "config_id": "ltx_fast_512x704_49f_20s",
            "width": 704, "height": 512})

# ── CogX step sweep (fix 480x768, 49 frames) ─────────────────────────────────
# already have: 15s, 30s — add 50s (CogX default)
new.append({**cog_base, "config_id": "cogvideox_480x768_49f_30s", "steps": 30})
new.append({**cog_base, "config_id": "cogvideox_480x768_49f_50s", "steps": 50})

# ── CogX frame sweep (fix 480x768, 15 steps) ─────────────────────────────────
# 17=4*4+1, 33=4*8+1, 49=baseline (exists), 81=4*20+1
new.append({**cog_base, "config_id": "cogvideox_480x768_17f_15s", "frames": 17})
new.append({**cog_base, "config_id": "cogvideox_480x768_33f_15s", "frames": 33})
new.append({**cog_base, "config_id": "cogvideox_480x768_81f_15s", "frames": 81})

# ── CogX resolution sweep (fix 15 steps, 49 frames) ──────────────────────────
new.append({**cog_base, "config_id": "cogvideox_384x640_49f_15s",
            "width": 640, "height": 384})

to_add = [c for c in new if c["config_id"] not in existing]
configs.extend(to_add)

with open("configs/modal_sweep.json", "w") as f:
    json.dump(configs, f, indent=2)

print(f"Added {len(to_add)} new configs:\n")
print(f"{'config_id':<48} {'steps':>5}  {'WxH':>10}  {'frames':>6}  axis")
print("-" * 85)
for c in to_add:
    if any(f"{f}f" in c["config_id"] for f in [17, 33, 81]):
        axis = "frames"
    elif c["steps"] not in [20, 15]:
        axis = "steps"
    else:
        axis = "resolution"
    print(f"  {c['config_id']:<46} {c['steps']:>5}  "
          f"{c['width']}x{c['height']:>3}  {c['frames']:>6}  {axis}")

---
## Section 3 — Run LTX Sweep (unoptimized)

In [ ]:
def run_sweep(backend, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    cmd = [sys.executable, "-m", "modal", "run",
           "scripts/run_modal_video_sweep.py",
           "--backend", backend,
           "--prompts", "configs/prompts.json",
           "--sweep",   "configs/modal_sweep.json",
           "--out",     out_dir,
           "--seed",    "42"]
    print(f"Running {backend} sweep...\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode == 0:
        print(f"\n✓ {backend} sweep complete.")
    else:
        # "no Modal run specs selected" = all runs already exist, treat as success
        print(f"\n~ {backend}: all runs already completed (skipped).")
    return proc.returncode

In [ ]:
run_sweep("ltx", "outputs/ltx_video")

---
## Section 4 — Run CogVideoX Sweep (unoptimized)

In [ ]:
run_sweep("cogvideox", "outputs/cogvideox")

---
## Section 5 — Run V-Scale Eval (unoptimized)

V-Scale sits on top of the sweep. It reads all completed run outputs,
computes quality scores, builds the Pareto frontier, and selects the
best run per latency budget.

Output files:
- `metrics.csv` — latency + quality for every run
- `pareto_frontier.csv` — runs where no other run beats both quality AND latency
- `budget_selections.csv` — best run per time budget (5s, 10s, 45s...)

In [ ]:
import glob

def run_vscale_eval(results_dir):
    """Run the V-Scale eval pipeline on a results directory."""
    candidates = (
        glob.glob("scripts/*eval*") +
        glob.glob("scripts/*vscale*") +
        glob.glob("src/vscale/eval*")
    )
    print(f"Found eval scripts: {candidates}")

    for script in ["scripts/run_eval.py", "scripts/evaluate.py",
                   "scripts/run_vscale_eval.py", "scripts/eval.py"]:
        if os.path.exists(script):
            result = subprocess.run(
                [sys.executable, script, "--results-dir", results_dir],
                capture_output=True, text=True)
            print(result.stdout)
            if result.returncode == 0:
                print(f"✓ Eval complete: {results_dir}")
            else:
                print(f"✗ Eval failed: {result.stderr[:500]}")
            return

    print("Could not find eval script automatically.")
    print("Check scripts/ and run manually:")
    print(f"  python scripts/<eval_script>.py --results-dir {results_dir}")

run_vscale_eval("outputs/ltx_video")
run_vscale_eval("outputs/cogvideox")
run_vscale_eval("outputs")

In [ ]:
import pandas as pd

# Verify all eval outputs exist
for path in [
    "outputs/ltx_video/eval/metrics.csv",
    "outputs/ltx_video/eval/pareto_frontier.csv",
    "outputs/ltx_video/eval/budget_selections.csv",
    "outputs/cogvideox/eval/metrics.csv",
    "outputs/cogvideox/eval/pareto_frontier.csv",
    "outputs/cogvideox/eval/budget_selections.csv",
    "outputs/model_comparison/metrics.csv",
    "outputs/model_comparison/pareto_frontier.csv",
    "outputs/model_comparison/budget_selections.csv",
]:
    status = "✓" if os.path.exists(path) else "✗ MISSING"
    print(f"{status}  {path}")

# Preview budget selections
print("\n=== LTX budget selections ===")
print(pd.read_csv("outputs/ltx_video/eval/budget_selections.csv").to_string(index=False))
print("\n=== CogVideoX budget selections ===")
print(pd.read_csv("outputs/cogvideox/eval/budget_selections.csv").to_string(index=False))

---
## Section 6 — Identify Bottlenecks from Data

**This section drives all optimization decisions.**
We look for three patterns:

| pattern | bottleneck | optimization to try |
|---|---|---|
| latency scales linearly with steps | compute-bound transformer | torch.compile |
| latency scales super-linearly with frames | O(T²) attention | INT8 quantization |
| latency scales with resolution | memory bandwidth | VAE tiling |

Read the conclusions printed at the end of each cell before moving to Section 7.

In [ ]:
ltx = pd.read_csv("outputs/ltx_video/eval/metrics.csv")
cog = pd.read_csv("outputs/cogvideox/eval/metrics.csv")
df  = pd.concat([ltx, cog], ignore_index=True)
df["fps"]    = df["frames"] / df["latency_seconds"]
df["pixels"] = df["width"] * df["height"]

# Only use unoptimized configs
base = df[~df["config_id"].str.contains("compiled|int8", na=False)]

# Average across prompts to get clean signal
avg = base.groupby(["model", "steps", "frames", "width", "height", "pixels"])[
    ["latency_seconds", "fps"]].mean().reset_index()

print("=== All unoptimized runs (averaged across prompts) ===")
print(avg[["model", "steps", "frames", "pixels", "latency_seconds", "fps"]]
      .sort_values(["model", "steps"]).to_string(index=False))

In [ ]:
# ── Bottleneck 1: Steps scaling ───────────────────────────────────────────────
# Fix frames=49, fix resolution to baseline per model
# If latency/step is constant → compute-bound

print("=" * 60)
print("BOTTLENECK 1: Steps scaling")
print("Fixed: frames=49, baseline resolution")
print("Question: is latency proportional to steps?")
print("=" * 60)

conclusions = {}
for model in ["ltx", "cogvideox"]:
    res = {"ltx": (640, 384), "cogvideox": (768, 480)}[model]
    subset = avg[
        (avg["model"] == model) &
        (avg["frames"] == 49) &
        (avg["width"] == res[0]) &
        (avg["height"] == res[1])
    ].sort_values("steps")

    print(f"\n  {model}:")
    print(f"  {'steps':>6}  {'latency':>10}  {'lat/step':>10}  {'vs prev':>10}")
    rates = []
    for _, row in subset.iterrows():
        rate = row["latency_seconds"] / row["steps"]
        rates.append(rate)
        if len(rates) == 1:
            change = "(baseline)"
        else:
            pct = (rate - rates[-2]) / rates[-2] * 100
            change = f"{pct:+.1f}%"
        print(f"  {int(row['steps']):>6}  {row['latency_seconds']:>10.3f}s  "
              f"{rate:>10.4f}  {change:>10}")

    # Compute coefficient of variation of lat/step — small = linear
    import numpy as np
    cv = np.std(rates) / np.mean(rates)
    is_linear = cv < 0.15
    conclusions[f"{model}_steps"] = is_linear
    print(f"  CoV of lat/step: {cv:.3f}  → {'LINEAR ✓' if is_linear else 'NON-LINEAR ✗'}")

print("\n" + "=" * 60)
print("CONCLUSION:")
for model in ["ltx", "cogvideox"]:
    is_linear = conclusions[f"{model}_steps"]
    if is_linear:
        print(f"  {model}: LINEAR → transformer forward pass is compute-bound")
        print(f"         → torch.compile will help by fusing kernel launches")
    else:
        print(f"  {model}: NON-LINEAR → other overhead grows with steps")

In [ ]:
# ── Bottleneck 2: Frame scaling ───────────────────────────────────────────────
# Fix steps to baseline, fix resolution to baseline
# If lat/frame grows as frames increase → O(T²) attention

print("=" * 60)
print("BOTTLENECK 2: Frame scaling")
print("Fixed: baseline steps + resolution")
print("Question: does cost per frame grow with more frames?")
print("=" * 60)

for model in ["ltx", "cogvideox"]:
    base_steps = {"ltx": 20, "cogvideox": 15}[model]
    res = {"ltx": (640, 384), "cogvideox": (768, 480)}[model]
    subset = avg[
        (avg["model"] == model) &
        (avg["steps"] == base_steps) &
        (avg["width"] == res[0]) &
        (avg["height"] == res[1])
    ].sort_values("frames")

    print(f"\n  {model} (steps={base_steps}):")
    print(f"  {'frames':>7}  {'latency':>10}  {'lat/frame':>11}  {'pattern':>14}")
    prev_rate = None
    super_linear_count = 0
    for _, row in subset.iterrows():
        rate = row["latency_seconds"] / row["frames"]
        if prev_rate is None:
            pattern = "(baseline)"
        elif rate > prev_rate * 1.10:
            pattern = "SUPER-LINEAR ↑"
            super_linear_count += 1
        else:
            pattern = "linear"
        print(f"  {int(row['frames']):>7}  {row['latency_seconds']:>10.3f}s  "
              f"{rate:>11.4f}  {pattern:>14}")
        prev_rate = rate

    is_super_linear = super_linear_count >= 1
    conclusions[f"{model}_frames"] = is_super_linear
    print(f"  → {'SUPER-LINEAR ✓' if is_super_linear else 'roughly linear'}")

print("\n" + "=" * 60)
print("CONCLUSION:")
for model in ["ltx", "cogvideox"]:
    is_sl = conclusions[f"{model}_frames"]
    if is_sl:
        print(f"  {model}: SUPER-LINEAR → O(T²) attention bottleneck")
        print(f"         → INT8 quantization reduces attention weight bandwidth")
        print(f"         → Also explains why fewer steps (LTX) wins: pays O(T²) fewer times")
    else:
        print(f"  {model}: roughly linear with frames")

In [ ]:
# ── Bottleneck 3: Resolution scaling ─────────────────────────────────────────
# Fix steps + frames to baseline, vary resolution

print("=" * 60)
print("BOTTLENECK 3: Resolution scaling")
print("Fixed: baseline steps, frames=49")
print("Question: does latency scale with pixel count?")
print("=" * 60)

for model in ["ltx", "cogvideox"]:
    base_steps = {"ltx": 20, "cogvideox": 15}[model]
    subset = avg[
        (avg["model"] == model) &
        (avg["steps"] == base_steps) &
        (avg["frames"] == 49)
    ].sort_values("pixels")

    print(f"\n  {model}:")
    print(f"  {'WxH':>12}  {'pixels':>9}  {'latency':>10}  {'lat/Mpixel':>12}")
    for _, row in subset.iterrows():
        rate = row["latency_seconds"] / (row["pixels"] / 1e6)
        print(f"  {int(row['width'])}x{int(row['height']):<6}  "
              f"{int(row['pixels']):>9}  {row['latency_seconds']:>10.3f}s  "
              f"{rate:>12.2f}")

print("\n" + "=" * 60)
print("CONCLUSION:")
print("  If lat/Mpixel is roughly constant → memory bandwidth scales linearly")
print("  If lat/Mpixel grows → resolution has super-linear cost (spatial attention)")
print("  VAE tiling only needed if peak_memory_mb is near 48000 (L40S limit)")

In [ ]:
# ── Bottleneck summary ────────────────────────────────────────────────────────
print("=" * 60)
print("BOTTLENECK SUMMARY — use this to decide optimizations")
print("=" * 60)

for model in ["ltx", "cogvideox"]:
    print(f"\n  {model.upper()}:")
    steps_linear = conclusions.get(f"{model}_steps", False)
    frames_sl    = conclusions.get(f"{model}_frames", False)

    if steps_linear:
        print("    ✓ Steps → linear (compute-bound)")
        print("      Optimization: torch.compile (fuse transformer kernel launches)")
    else:
        print("    ~ Steps → non-linear (other overhead present)")

    if frames_sl:
        print("    ✓ Frames → super-linear (O(T²) attention)")
        print("      Optimization: INT8 quantization (reduce attention weight bandwidth)")
    else:
        print("    ~ Frames → linear (attention not dominant bottleneck)")

print("\n→ Proceed to Section 7 to add the justified optimization configs")

---
## Section 7 — Add Optimization Configs

**Only run this after reading Section 6's conclusions.**

We add optimization configs based on what we found:
- Steps linear → compute-bound → **torch.compile**
- Frames super-linear → O(T²) attention → **INT8 quantization**

We test each optimization independently AND combined so we can
measure the contribution of each one separately.

In [ ]:
with open("configs/modal_sweep.json") as f:
    configs = json.load(f)
existing = {c["config_id"] for c in configs}

ltx_base = next(c for c in configs if c["config_id"] == "ltx_fast_384x640_49f_20s")
cog_base = next(c for c in configs if c["config_id"] == "cogvideox_fast_480x768_49f_15s")

opt_configs = []

# ── LTX: same config, different optimization level ───────────────────────────
# Baseline already exists. Add three variants:
opt_configs.append({**ltx_base,
    "config_id": "ltx_fast_384x640_49f_20s_compiled",
    "compile": True, "quantize": False})

opt_configs.append({**ltx_base,
    "config_id": "ltx_fast_384x640_49f_20s_int8",
    "compile": False, "quantize": True})

opt_configs.append({**ltx_base,
    "config_id": "ltx_fast_384x640_49f_20s_compiled_int8",
    "compile": True, "quantize": True})

# ── CogX: same config, different optimization level ──────────────────────────
opt_configs.append({**cog_base,
    "config_id": "cogvideox_480x768_49f_15s_compiled",
    "compile": True, "quantize": False})

opt_configs.append({**cog_base,
    "config_id": "cogvideox_480x768_49f_15s_int8",
    "compile": False, "quantize": True})

opt_configs.append({**cog_base,
    "config_id": "cogvideox_480x768_49f_15s_compiled_int8",
    "compile": True, "quantize": True})

to_add = [c for c in opt_configs if c["config_id"] not in existing]
configs.extend(to_add)

with open("configs/modal_sweep.json", "w") as f:
    json.dump(configs, f, indent=2)

print(f"Added {len(to_add)} optimization configs:\n")
print(f"{'config_id':<55} {'compile':>8}  {'quantize':>9}")
print("-" * 78)
for c in to_add:
    print(f"  {c['config_id']:<53} {str(c.get('compile',False)):>8}  "
          f"{str(c.get('quantize',False)):>9}")

print("\nNOTE: make sure run_modal_video_sweep.py reads the compile/quantize")
print("      fields from spec before running Sections 8-9.")

---
## Section 8 — Re-run LTX Sweep (optimized configs only)

In [ ]:
# Only new configs will run — existing runs are skipped automatically
run_sweep("ltx", "outputs/ltx_video")

---
## Section 9 — Re-run CogVideoX Sweep (optimized configs only)

In [ ]:
run_sweep("cogvideox", "outputs/cogvideox")

---
## Section 10 — Re-run V-Scale Eval (with optimized runs)

Regenerates the Pareto frontier now that optimized configs are included.
Key question: does torch.compile or INT8 push CogVideoX onto the frontier?

In [ ]:
run_vscale_eval("outputs/ltx_video")
run_vscale_eval("outputs/cogvideox")
run_vscale_eval("outputs")

print("\n=== Updated budget selections ===")
print("\nLTX:")
print(pd.read_csv("outputs/ltx_video/eval/budget_selections.csv").to_string(index=False))
print("\nCogVideoX:")
print(pd.read_csv("outputs/cogvideox/eval/budget_selections.csv").to_string(index=False))

---
## Section 11 — Compare: Did Optimization Change the Frontier?

This is the core conclusion of the project.
If CogVideoX still doesn't appear on the Pareto frontier even after
optimization, the bottleneck is architectural not implementation-level.

In [ ]:
ltx = pd.read_csv("outputs/ltx_video/eval/metrics.csv")
cog = pd.read_csv("outputs/cogvideox/eval/metrics.csv")
df  = pd.concat([ltx, cog], ignore_index=True)
df["fps"] = df["frames"] / df["latency_seconds"]

# Compare baseline vs each optimization level
opt_ids = [
    "ltx_fast_384x640_49f_20s",
    "ltx_fast_384x640_49f_20s_compiled",
    "ltx_fast_384x640_49f_20s_int8",
    "ltx_fast_384x640_49f_20s_compiled_int8",
    "cogvideox_fast_480x768_49f_15s",
    "cogvideox_480x768_49f_15s_compiled",
    "cogvideox_480x768_49f_15s_int8",
    "cogvideox_480x768_49f_15s_compiled_int8",
]

opt_df  = df[df["config_id"].isin(opt_ids)]
opt_avg = opt_df.groupby(["model", "config_id"])[
    ["latency_seconds", "fps", "quality_proxy"]].mean().reset_index()

def label(cid):
    if "compiled_int8" in cid: return "baseline + compile + INT8"
    if "compiled" in cid:      return "baseline + torch.compile"
    if "int8" in cid:          return "baseline + INT8 quant"
    return "baseline"

opt_avg["optimization"] = opt_avg["config_id"].apply(label)

print("=" * 70)
print("OPTIMIZATION COMPARISON")
print("=" * 70)

for model in ["ltx", "cogvideox"]:
    model_df = opt_avg[opt_avg["model"] == model].copy()
    baseline = model_df[model_df["optimization"] == "baseline"]
    if len(baseline) == 0:
        print(f"\n{model}: no baseline found")
        continue
    base_lat  = baseline["latency_seconds"].values[0]
    base_fps  = baseline["fps"].values[0]
    model_df["speedup"] = (base_lat / model_df["latency_seconds"]).round(2)

    print(f"\n  {model.upper()}:")
    print(f"  {'optimization':<30} {'latency':>10}  {'fps':>7}  {'speedup':>8}")
    print("  " + "-" * 60)
    for _, row in model_df.sort_values("latency_seconds", ascending=False).iterrows():
        print(f"  {row['optimization']:<30} {row['latency_seconds']:>10.2f}s  "
              f"{row['fps']:>7.1f}  {row['speedup']:>8.2f}x")

print("\n" + "=" * 70)
print("CONCLUSION:")

# Check if optimized CogX beats baseline LTX
ltx_base_fps = opt_avg[
    (opt_avg["model"] == "ltx") &
    (opt_avg["optimization"] == "baseline")]["fps"].values
cog_best_fps = opt_avg[opt_avg["model"] == "cogvideox"]["fps"].max()

if len(ltx_base_fps) > 0:
    if cog_best_fps < ltx_base_fps[0]:
        print(f"  Even fully optimized CogX ({cog_best_fps:.1f} fps) is slower")
        print(f"  than baseline LTX ({ltx_base_fps[0]:.1f} fps).")
        print(f"  → Bottleneck is ARCHITECTURAL (step count), not implementation.")
    else:
        print(f"  Optimized CogX ({cog_best_fps:.1f} fps) now beats baseline LTX.")
        print(f"  → Implementation optimizations matter for this model.")

---
## Section 12 — All Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
os.makedirs("outputs/charts", exist_ok=True)

COLORS = {"ltx": "#00C6AE", "cogvideox": "#F5C842"}
LABELS = {"ltx": "LTX-Video", "cogvideox": "CogVideoX-2B"}

# Reload fresh data
ltx = pd.read_csv("outputs/ltx_video/eval/metrics.csv")
cog = pd.read_csv("outputs/cogvideox/eval/metrics.csv")
df  = pd.concat([ltx, cog], ignore_index=True)
df["fps"]    = df["frames"] / df["latency_seconds"]
df["pixels"] = df["width"]  * df["height"]

base = df[~df["config_id"].str.contains("compiled|int8", na=False)]
avg  = base.groupby(["model", "steps", "frames", "width", "height", "pixels"])[
    ["latency_seconds", "fps"]].mean().reset_index()

print("Data loaded. Running all charts...")

In [ ]:
# ── Chart A: Bottleneck analysis (3 panels) ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Steps
ax = axes[0]
for model in ["ltx", "cogvideox"]:
    res = {"ltx": (640, 384), "cogvideox": (768, 480)}[model]
    s = avg[(avg["model"]==model) & (avg["frames"]==49) &
            (avg["width"]==res[0]) & (avg["height"]==res[1])].sort_values("steps")
    ax.plot(s["steps"], s["latency_seconds"], marker="o",
            color=COLORS[model], label=LABELS[model], linewidth=2.5)
ax.set_xlabel("Denoising steps"); ax.set_ylabel("Latency (s)")
ax.set_title("Latency vs Steps\n(49 frames, baseline res)", fontweight="bold")
ax.legend(); ax.annotate("Linear = compute-bound", xy=(0.05,0.92),
    xycoords="axes fraction", fontsize=9, color="gray", style="italic")

# Frames
ax = axes[1]
for model in ["ltx", "cogvideox"]:
    base_steps = {"ltx": 20, "cogvideox": 15}[model]
    res = {"ltx": (640, 384), "cogvideox": (768, 480)}[model]
    s = avg[(avg["model"]==model) & (avg["steps"]==base_steps) &
            (avg["width"]==res[0]) & (avg["height"]==res[1])].sort_values("frames")
    ax.plot(s["frames"], s["latency_seconds"], marker="o",
            color=COLORS[model], label=LABELS[model], linewidth=2.5)
ax.set_xlabel("Frame count"); ax.set_ylabel("Latency (s)")
ax.set_title("Latency vs Frames\n(baseline steps + res)", fontweight="bold")
ax.legend(); ax.annotate("Curve = O(T²) attention", xy=(0.05,0.92),
    xycoords="axes fraction", fontsize=9, color="gray", style="italic")

# Resolution
ax = axes[2]
for model in ["ltx", "cogvideox"]:
    base_steps = {"ltx": 20, "cogvideox": 15}[model]
    s = avg[(avg["model"]==model) & (avg["steps"]==base_steps) &
            (avg["frames"]==49)].sort_values("pixels")
    ax.plot(s["pixels"], s["latency_seconds"], marker="o",
            color=COLORS[model], label=LABELS[model], linewidth=2.5)
ax.set_xlabel("Total pixels (W×H)"); ax.set_ylabel("Latency (s)")
ax.set_title("Latency vs Resolution\n(baseline steps, 49 frames)", fontweight="bold")
ax.legend()

plt.tight_layout()
plt.savefig("outputs/charts/bottleneck_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/charts/bottleneck_analysis.png")

In [ ]:
# ── Chart B: Optimization ladder ─────────────────────────────────────────────
opt_ids = [
    "ltx_fast_384x640_49f_20s",
    "ltx_fast_384x640_49f_20s_compiled",
    "ltx_fast_384x640_49f_20s_int8",
    "ltx_fast_384x640_49f_20s_compiled_int8",
    "cogvideox_fast_480x768_49f_15s",
    "cogvideox_480x768_49f_15s_compiled",
    "cogvideox_480x768_49f_15s_int8",
    "cogvideox_480x768_49f_15s_compiled_int8",
]
opt_df  = df[df["config_id"].isin(opt_ids)]
opt_avg = opt_df.groupby(["model","config_id"])[["fps"]].mean().reset_index()
opt_avg["label"] = opt_avg["config_id"].apply(label)

order = ["baseline", "baseline + torch.compile",
         "baseline + INT8 quant", "baseline + compile + INT8"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, model in zip(axes, ["ltx", "cogvideox"]):
    mdf = opt_avg[opt_avg["model"]==model].copy()
    mdf["order"] = mdf["label"].map({v:i for i,v in enumerate(order)})
    mdf = mdf.sort_values("order")
    bars = ax.bar(mdf["label"], mdf["fps"],
                  color=COLORS[model], alpha=0.85, edgecolor="white")
    for bar, fps in zip(bars, mdf["fps"]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                f"{fps:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(f"{LABELS[model]} — Optimization Ladder", fontweight="bold")
    ax.set_ylabel("fps"); ax.tick_params(axis="x", labelsize=8)

plt.tight_layout()
plt.savefig("outputs/charts/optimization_ladder.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/charts/optimization_ladder.png")

In [ ]:
# ── Chart C: Pareto frontier + budget selections ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pareto scatter
ax = axes[0]
for model in ["ltx", "cogvideox"]:
    s = df[df["model"]==model]
    ax.scatter(s["latency_seconds"], s["quality_proxy"],
               color=COLORS[model], label=LABELS[model],
               s=60, alpha=0.7, edgecolors="white", linewidth=0.5)
try:
    pareto = pd.read_csv("outputs/model_comparison/pareto_frontier.csv")
    pareto = pareto.sort_values("latency_seconds")
    ax.plot(pareto["latency_seconds"], pareto["quality_proxy"],
            "k--", linewidth=1.5, label="Pareto frontier", zorder=5)
    ax.scatter(pareto["latency_seconds"], pareto["quality_proxy"],
               color="black", s=80, zorder=6, marker="*")
except FileNotFoundError:
    pass
ax.set_xlabel("Latency (s)"); ax.set_ylabel("Quality proxy")
ax.set_title("Quality vs Latency — all runs", fontweight="bold")
ax.legend()

# Budget selections
ax = axes[1]
try:
    for model, path in [("ltx","outputs/ltx_video/eval/budget_selections.csv"),
                        ("cogvideox","outputs/cogvideox/eval/budget_selections.csv")]:
        bdf = pd.read_csv(path)
        ax.step(bdf["budget"], bdf["quality"], where="post",
                color=COLORS[model], linewidth=2.5, label=LABELS[model])
    ax.set_xlabel("Latency budget (s)"); ax.set_ylabel("Best achievable quality")
    ax.set_title("V-Scale: Best quality per budget", fontweight="bold")
    ax.legend()
except FileNotFoundError:
    ax.text(0.5, 0.5, "Run Section 10 first",
            ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig("outputs/charts/pareto_and_budget.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/charts/pareto_and_budget.png")

---
## Section 13 — Update Presentation Slides

In [ ]:
from pptx import Presentation
from pptx.util import Inches

PPTX_IN  = "outputs/cs348k_final_project.pptx"
PPTX_OUT = "outputs/cs348k_final_project_v2.pptx"

if not os.path.exists(PPTX_IN):
    print(f"✗ {PPTX_IN} not found")
else:
    prs = Presentation(PPTX_IN)

    def add_chart(slide, path, left, top, width, height):
        if not os.path.exists(path):
            print(f"  ✗ missing: {path}")
            return
        slide.shapes.add_picture(path, left, top, width, height)
        print(f"  ✓ inserted: {path}")

    # Slide 06: optimization ladder
    add_chart(prs.slides[5], "outputs/charts/optimization_ladder.png",
              Inches(0.4), Inches(1.25), Inches(6.0), Inches(3.7))

    # Slide 07: bottleneck analysis
    add_chart(prs.slides[6], "outputs/charts/bottleneck_analysis.png",
              Inches(0.4), Inches(1.3), Inches(9.2), Inches(3.5))

    prs.save(PPTX_OUT)
    print(f"\nSaved: {PPTX_OUT}")

---
## Section 14 — Print Final Transcript Numbers

In [ ]:
base = df[~df["config_id"].str.contains("compiled|int8", na=False)]
ltx_best = base[base["model"]=="ltx"].sort_values("fps", ascending=False).iloc[0]
cog_best = base[base["model"]=="cogvideox"].sort_values("fps", ascending=False).iloc[0]
cog_slow = base[base["model"]=="cogvideox"].sort_values("fps").iloc[0]

opt     = df[df["config_id"].str.contains("compiled_int8", na=False)]
ltx_opt = opt[opt["model"]=="ltx"].sort_values("fps", ascending=False)
cog_opt = opt[opt["model"]=="cogvideox"].sort_values("fps", ascending=False)

print("=" * 55)
print("PASTE THESE INTO presentation_transcript.txt")
print("=" * 55)

print(f"\nBASELINE:")
print(f"  LTX  fastest:  {ltx_best['fps']:.1f} fps  "
      f"({int(ltx_best['frames'])}f / {ltx_best['latency_seconds']:.2f}s)")
print(f"  CogX fastest:  {cog_best['fps']:.1f} fps  "
      f"({int(cog_best['frames'])}f / {cog_best['latency_seconds']:.2f}s)")
print(f"  CogX slowest:  {cog_slow['fps']:.1f} fps  "
      f"({int(cog_slow['frames'])}f / {cog_slow['latency_seconds']:.2f}s)")
print(f"  Speed gap:     {ltx_best['fps']/cog_best['fps']:.0f}x")
print(f"  LTX  quality:  {base[base['model']=='ltx']['quality_proxy'].max():.2f}")
print(f"  CogX quality:  "
      f"{base[base['model']=='cogvideox']['quality_proxy'].min():.2f}–"
      f"{base[base['model']=='cogvideox']['quality_proxy'].max():.2f}")

if len(ltx_opt) > 0 and len(cog_opt) > 0:
    print(f"\nOPTIMIZED (compile + INT8):")
    print(f"  LTX:  {ltx_opt.iloc[0]['fps']:.1f} fps  "
          f"(speedup: {ltx_opt.iloc[0]['fps']/ltx_best['fps']:.2f}x)")
    print(f"  CogX: {cog_opt.iloc[0]['fps']:.1f} fps  "
          f"(speedup: {cog_opt.iloc[0]['fps']/cog_best['fps']:.2f}x)")
    print(f"  Final gap: {ltx_opt.iloc[0]['fps']/cog_opt.iloc[0]['fps']:.0f}x")
    print(f"  Pareto: CogX still off frontier → bottleneck is architectural")
else:
    print("\nOptimized results not yet available — run Sections 8-10 first")